# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(url)

# Access and display the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Authors: {[getattr(author, 'name', str(author)) for author in getattr(metadata, 'author', [])]}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities (record sets, fields, columns, etc.) are referenced by their `@id` identifier.

In [ ]:
# List all record sets in the dataset with their @id and name
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"@id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")
    
    # List the fields for each record set
    for rs in record_sets:
        print(f"\nFields for record set @id {rs['@id']} ({rs.get('name', 'unnamed')}):")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"  - Field @id: {field.get('@id', '(no id)')} | Name: {field.get('name', '(no name)')}")
            else:
                print(f"  - Field @id: {field}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

_**All elements are referenced by their `@id`.**_

In [ ]:
# If there are no record sets, skip loading
if not record_sets:
    print("There are no record sets to extract.")
else:
    # Extract all record sets by @id
    record_set_ids = [rs['@id'] for rs in record_sets]
    print("Extracting data from record sets:", record_set_ids)
    dataframes = {}
    for record_set_id in record_set_ids:
        try:
            # records() expects record_set as @id
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded record set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Could not load record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

We'll apply basic data processing steps: filtering numeric fields, normalizing them, and grouping by categorical fields, referencing all fields and columns by `@id`.

In [ ]:
# This cell demonstrates EDA for the first loaded record set.
import numpy as np

if not dataframes:
    print("No data available for EDA.")
else:
    # Select the first record set and its dataframe
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Performing EDA on record set: {selected_record_set_id}")

    # Display columns and data types
    print("Columns and types:")
    print(df.dtypes)

    # Identify a likely numeric field by looking for numeric column (float/int, not bool/object)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col])]

    if not numeric_fields:
        print("No numeric fields available to analyze.")
    else:
        numeric_field_id = numeric_fields[0]  # Use the first one found
        print(f"Using numeric field (by @id): {numeric_field_id}")

        # Example: filter rows with value > threshold
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping: look for any categorical (object) field
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping data by '{group_field_id}' (@id)")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped means:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if not dataframes:
    print("No data available for visualization.")
elif not numeric_fields:
    print("No numeric field available to visualize.")
else:
    # Distribution of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df available, barplot
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 dataset via its Croissant schema, reviewed available record sets and fields (with all referencing by `@id`), extracted tabular data for exploration, performed basic EDA including filtering and normalization on numeric fields, and visualized data distributions. These steps can be adapted for deeper analysis, statistical modeling, and domain-specific applications relevant to the dataset's focus on knowledge adoption in rangeland management in Northern Kenya.